In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import platform
import koreanize_matplotlib

# df_merge

In [ ]:
df = pd.read_csv('./data/full_merged_v2.csv')

In [ ]:
# drop_cols = ['device', 'identity_failure_reason', 'mp_sent_by_lib_version', 'page_url']

In [ ]:
#df.drop(drop_cols, axis=1, inplace=True, errors='ignore')

### trigger, position, platform 셀 복구

In [ ]:
df3 = pd.read_json('data/3. 이벤트 데이터.json', lines=True)
df3['user_id_str'] = df3['user_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else x)
if 'properties' in df3.columns:
    props_df3 = pd.json_normalize(df3['properties'])
    for col in ['distinct_id', 'user_id']:
        if col in props_df3.columns:
            props_df3 = props_df3.drop(col, axis=1)
    df3 = pd.concat([df3.drop('properties', axis=1), props_df3], axis=1)

In [ ]:
# df['trigger'] = df3['trigger']

In [ ]:
#df['position'] = df3['position']

In [ ]:
df3['platform'].value_counts()

In [ ]:
df3[df3['platform'].notnull()]['position'].value_counts()

In [ ]:
df3[df3['position'] == 'popup']['platform'].value_counts()

In [ ]:
df['platform'] = df3['platform']

In [ ]:
df.to_csv('./data/full_merged_v3.csv', index=False)

# df3

In [ ]:
df3 = pd.read_json('data/3. 이벤트 데이터.json', lines=True)

In [ ]:
df3['user_id'] = df3['user_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else x)

In [ ]:
if 'properties' in df3.columns:
    props_df3 = pd.json_normalize(df3['properties'])
    for col in ['distinct_id', 'user_id']:
        if col in props_df3.columns:
            props_df3 = props_df3.drop(col, axis=1)
    df3 = pd.concat([df3.drop('properties', axis=1), props_df3], axis=1)

### 추가 삭제할 컬럼
'device', 'identity_failure_reason', 'mp_sent_by_lib_version', 'page_url'

In [ ]:
drop_cols_df3 = [
    'browser_is_whale', 'browser_is_chrome', 'browser_is_edge',
    'email', 'name', 'browser_version', 'insert_id',
    'screen_height', 'screen_width', 'search_engine', 'lib_version',
    'mp_api_endpoint', 'mp_lib', 'mp_loader', 'extension_version',
    'llm_provider', 'llm_version', 'position', 'platform', 'trigger','device', 'identity_failure_reason', 'mp_sent_by_lib_version', 'page_url'
]

In [ ]:
df3.drop(drop_cols_df3, axis=1, inplace=True, errors='ignore')

In [ ]:
df3

In [ ]:
df3.columns

In [ ]:
df3['position'].value_counts()

### environment 확인 -> 결론 : 유지

In [ ]:
df3['environment'].value_counts()

In [ ]:
df3[df3['environment'].isna()]['event_name'].value_counts()

In [ ]:
df3[df3['environment'].isna()]['mp_country_code'].value_counts()

In [ ]:
df3[df3['environment'].isna()]['mp_sent_by_lib_version'].value_counts()

In [ ]:
df3[df3['environment'].isna()]['referring_domain'].value_counts()

위 정보만으로는 개발자의 이벤트를 식별할만한 정보를 찾기 어려웠다.  
environment가 'production'인 그룹과 'null'인 그룹이 통계적으로 서로 다른 집단인지 확인해보자.

In [ ]:
df_prod = df3[df3['environment'] == 'production']
df_null = df3[df3['environment'].isna()]

In [ ]:
# 'production' 그룹의 이벤트 비율
print(df_prod['event_name'].value_counts(normalize=True))

# 'null' 그룹의 이벤트 비율
print(df_null['event_name'].value_counts(normalize=True))

In [ ]:
print(df_prod['os'].value_counts(normalize=True))
print(df_null['os'].value_counts(normalize=True))

print(df_prod['browser'].value_counts(normalize=True))
print(df_null['browser'].value_counts(normalize=True))

In [ ]:
print(df_prod['initial_referring_domain'].value_counts().head(10))
print(df_null['initial_referring_domain'].value_counts().head(10))

두 그룹은 완전히 다른 집단이다.  
특이하게도, Chrome 사용자가 거의 대부분을 차지했다.  
Chrome 확장자를 사용한 이벤트에서 체계적으로 누락된것으로 추측해볼 수 있다.

### 'device' 확인 -> 결론 : 삭제, mobile이라는 정보는 os에 들어있음,

데이터의 개수가 비교적 작다.  
특히 iPhone과 iPad의 구분은, Android는 기기 구분이 없기 때문에 변수의 공정성이 깨져서 데이터 일관성이 부족하다.  

In [ ]:
df3['device'].value_counts()

device가 결측치이면서, os가 'Android, iOS' 인 데이터는 없다.  
즉,  
device에 있다면 반드시 os는 'Android, iOS'이다.  
device에 없다면 반드시 os는 PC의 os들이다.  

In [ ]:
chk1 = df3[df3['device'].notnull()]['os'].value_counts()
chk1

In [ ]:
chk2 = df3[df3['device'].isna()]['os'].value_counts()
chk2

전체 열은 62,097개인것을 확인했다.

In [ ]:
chk1.sum() + chk2.sum() + df3['os'].isnull().sum()

### feature engineering : is_pc -> 결론 : 데이터 필드에 추가
device와 os 컬럼을 고려

In [ ]:
df3['os'].value_counts()

전체 6만건의 데이터중, iOS 사용자는 12명에 불과하다.  
머신러닝 과적합 방지를 위해 모바일로 통합한다.

In [ ]:
# 1. OS 범주화 매핑 정의
# 유사한 OS끼리 묶어 마케팅 페르소나를 명확히 합니다.
os_mapping = {
    'Windows': 'Windows',
    'Win32': 'Windows',      # 32비트 윈도우도 결국 Windows
    'Mac OS X': 'Mac',
    'MacIntel': 'Mac',       # 인텔칩 맥도 결국 Mac
    'Chrome OS': 'Chrome OS',# 별도 분류 (교육용/가벼운 노트북 시장)
    'Linux': 'Linux',        # 별도 분류 (개발자군일 확률 높음)
    'Android': 'Mobile',     # 모바일 통합
    'iOS': 'Mobile'          # 모바일 통합 (데이터가 너무 적어 iOS/Android 구분 불필요)
}

In [ ]:
# df3가 데이터프레임 이름이라고 가정합니다.
# map 함수를 사용하여 os 컬럼을 'os_category'로 단순화합니다.
df3['os'] = df3['os'].map(os_mapping)

In [ ]:
# 2. is_PC 변수 생성
# Mobile로 분류되지 않은 모든 것을 PC로 간주합니다.
# (Windows, Mac, Chrome OS, Linux -> 1, Mobile -> 0)
df3['is_pc'] = np.where(df3['os'] == 'Mobile', 0, 1)

### 'identity_failure_reason' 확인 -> 결론 : 제거
이 데이터는 "공용 기기(PC방, 가족 공용 태블릿) 사용" 또는 "앱 삭제 후 재설치 과정에서 ID 매핑 꼬임" 등으로 인해 발생했을 가능성이 큽니다.

분석 관점: 이 로그들은 사용자 행동 데이터가 '유실'된 것이 아니라, '잘못된 병합을 방지'한 기록입니다. 해당 사용자의 행동은 새로운 익명 ID로 분리되어 별도로 집계되었거나, 기존 ID로 귀속되었을 것입니다. 분석 결과에 치명적인 오류를 주지는 않으므로, 이 에러가 발생한 행을 통계에서 제외하거나 Unknown으로 처리해도 무방합니다.

In [ ]:
df3['identity_failure_reason'].value_counts()

In [ ]:
df3[df3['identity_failure_reason'].notnull()]

### 리퍼럴 도메인에 대한 탐색 -> 결론 : 유지

- 'referrer', 'referring_domain'에 대한 결측치가 높다.
-  혹시 $direct로 들어왔을경우, 없는 것이 아닌가?

In [ ]:
target_cols = ['referrer', 'referring_domain', 'initial_referrer', 'initial_referring_domain']

# 1. 각 컬럼의 기본 통계 정보 (데이터 타입, 결측치 수, 고유값 수)
print("## 1. 컬럼별 요약 통계")
summary = pd.DataFrame({
    'Null_Count': df3[target_cols].isnull().sum(),
    'Null_Ratio(%)': (df3[target_cols].isnull().sum() / len(df3)) * 100,
    'Unique_Count': df3[target_cols].nunique()
})
print(summary)
print("\n" + "="*50 + "\n")

# 2. 실제 데이터 샘플 확인 (상위 5개)
# URL의 쿼리 스트링(물음표 뒤의 내용) 유무를 확인하기 위함.
print("## 2. 데이터 샘플 (상위 5행)")
print(df3[target_cols].head())
print("\n" + "="*50 + "\n")

# 3. 도메인과 URL의 관계 확인
# referrer가 있는데 referring_domain이 다른 경우가 있는지, 혹은 도메인이 URL에 포함되지 않는 예외가 있는지 확인
print("## 3. 도메인 매핑 샘플 확인 (랜덤 5개)")
# 비어있지 않은 데이터 중에서 랜덤 샘플링
non_null_sample = df3.dropna(subset=['referrer', 'referring_domain']).sample(n=5, random_state=42)
for index, row in non_null_sample.iterrows():
    print(f"Referrer: {row['referrer']}")
    print(f"Domain  : {row['referring_domain']}")
    print("-" * 30)

In [ ]:
df3['referrer'].value_counts()

In [ ]:
df3['referring_domain'].value_counts()

In [ ]:
df3['initial_referrer'].value_counts()

In [ ]:
df3[df3['referring_domain'].notnull()]['initial_referrer'].value_counts()

In [ ]:
df3[df3['referrer'].isna()]['initial_referrer'].value_counts()

# 오프라인에서 직접
# 시간대를 통해 데이터 수집이 안정화되는 것 또한 봐야

In [ ]:
df3[df3['referrer'].isna()]['initial_referring_domain'].value_counts()

In [ ]:
df3[df3['referring_domain'].isna()]['initial_referrer'].value_counts()

In [ ]:
df3[df3['referring_domain'].isna()]['initial_referring_domain'].value_counts()

In [ ]:
df3[df3['initial_referrer'].isna()]['referring_domain'].value_counts()

In [ ]:
df3[df3['initial_referrer'].notnull()]['referring_domain'].value_counts()

In [ ]:
df3['event_name'].value_counts()

In [ ]:
df3[df3['event_name'] == 'pageview_ad inflow']['initial_referrer'].value_counts()

In [ ]:
df3[df3['event_name'] == 'pageview_ad inflow']['referring_domain'].value_counts()

### utm 데이터 확인

In [ ]:
utm_cols = []
for col in df3.columns:
    if 'utm' in col:
        utm_cols.append(col)

utm_cols

In [ ]:
df[df3['utm_campaign'].notnull()]['referring_domain'].value_counts()

### 'event_name' 확인 -> 결론 : 데이터 필드 시트에 추가

In [ ]:
df3['event_name'].value_counts()

### gclid 확인 -> 결론 : 유지

In [ ]:
df3['gclid'].describe()

In [ ]:
df3['gclid'].value_counts()

In [ ]:
df3[df3['gclid'].notnull()]['event_name'].value_counts()

### 'mp_sent_by_lib_version' 확인 -> 결론 : 제거

In [ ]:
df3['mp_sent_by_lib_version'].value_counts()